In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# Load processed data from Stage 2
df = pd.read_csv("/content/drive/MyDrive/Frailty_Grip_Strength_Analysis/Data_Clean/frailty_processed.csv")

print(df)

   Height  Weight  Age  Grip strength Frailty  Height_m  Weight_kg    BMI  \
0    65.8     112   30             30       N   1.67132  50.802345  18.19   
1    71.5     136   19             31       N   1.81610  61.688562  18.70   
2    69.4     153   45             29       N   1.76276  69.399633  22.33   
3    68.2     142   22             28       Y   1.73228  64.410117  21.46   
4    67.8     144   29             24       Y   1.72212  65.317301  22.02   
5    68.7     123   50             26       N   1.74498  55.791862  18.32   
6    69.8     141   51             22       Y   1.77292  63.956524  20.35   
7    70.1     136   23             20       Y   1.78054  61.688562  19.46   
8    67.9     112   17             19       N   1.72466  50.802345  17.08   
9    66.8     120   39             31       N   1.69672  54.431084  18.91   

  AgeGroup  
0    30–45  
1      <30  
2    30–45  
3      <30  
4      <30  
5    46–60  
6    46–60  
7      <30  
8      <30  
9    30–45  


# **Categorical → Numeric Encoding**
# Binary Encoding: Frailty Y→1, N→0 **bold text**

In [3]:
# PART c(i) - Binary encode Frailty column (Y=1, N=0), stored as int8

df['Frailty_binary'] = df['Frailty'].map({'Y': 1, 'N': 0}).astype('int8')
print(df[['Frailty', 'Frailty_binary']])

  Frailty  Frailty_binary
0       N               0
1       N               0
2       N               0
3       Y               1
4       Y               1
5       N               0
6       Y               1
7       Y               1
8       N               0
9       N               0


In [4]:
# PART c(ii) - One-hot encode AgeGroup so each category becomes its own binary column

df = pd.concat([df, pd.get_dummies(df['AgeGroup'], prefix='AgeGroup')], axis=1)
df['AgeGroup_>60'] = False

print(df[['AgeGroup', 'AgeGroup_<30', 'AgeGroup_30–45', 'AgeGroup_46–60', 'AgeGroup_>60']])

  AgeGroup  AgeGroup_<30  AgeGroup_30–45  AgeGroup_46–60  AgeGroup_>60
0    30–45         False            True           False         False
1      <30          True           False           False         False
2    30–45         False            True           False         False
3      <30          True           False           False         False
4      <30          True           False           False         False
5    46–60         False           False            True         False
6    46–60         False           False            True         False
7      <30          True           False           False         False
8      <30          True           False           False         False
9    30–45         False            True           False         False


# **EDA & Reporting**
# **Summary Table: mean, median, std for numeric columns**

In [5]:
# PART d(I) - Compute summary statistics for all numeric columns

frailty_numeric_cols = df.select_dtypes(include='number').columns.tolist()
frailty_summary = df[frailty_numeric_cols].agg(['mean', 'median', 'std']).round(4)

print(frailty_summary)

         Height    Weight      Age  Grip strength  Height_m  Weight_kg  \
mean    68.6000  131.9000  32.5000        26.0000    1.7424    59.8288   
median  68.4500  136.0000  29.5000        27.0000    1.7386    61.6886   
std      1.6707   14.2318  12.8604         4.5216    0.0424     6.4554   

           BMI  Frailty_binary  
mean    19.682          0.4000  
median  19.185          0.0000  
std      1.781          0.5164  


In [7]:
# Save summary table to reports/findings.md
report_path = "/content/drive/MyDrive/Frailty_Grip_Strength_Analysis/Report/findings.md"

with open(report_path, 'w') as f:
    f.write("# Frailty Dataset - Findings Report\n\n")
    f.write("## Summary\n\n")
    f.write(frailty_summary.to_markdown())

print("Summary saved to findings.md!")

Summary saved to findings.md!


In [11]:
# PART d(II) - Compute correlation between Grip strength and Frailty_binary
# A negative value means as Grip strength increases, Frailty decreases.

grip_frailty_corr = df['Grip strength'].corr(df['Frailty_binary'])
print(f"Correlation (Grip_kg ↔ Frailty_binary): {grip_frailty_corr:.4f}")

Correlation (Grip_kg ↔ Frailty_binary): -0.4759


In [12]:
# d(II) - Append correlation report to findings.md
with open(report_path, 'a') as f:
    f.write("\n\n## Correlation: Grip Strength ↔ Frailty\n\n")
    f.write(f"**Pearson Correlation (Grip_kg ↔ Frailty_binary): {grip_frailty_corr:.4f}**\n\n")
    f.write(f"**Interpretation:** A correlation of {grip_frailty_corr:.4f} indicates a moderate negative relationship.\n")
    f.write("Higher grip strength is associated with lower frailty (N=0).\n")
    f.write("Lower grip strength is associated with higher frailty (Y=1).\n")
    f.write("This supports the study hypothesis that grip strength predicts frailty.\n")

print("Correlation report saved to findings.md!")

Correlation report saved to findings.md!
